In [1]:
# =============================================================================
# STEP 7c - THRESHOLD-REFERENCED MONITOR
#
# Two failed designs led here, and both failures were informative.
#
#   nb46, RANK-BASED: scores classes by within-environment percentile, so some
#   class always ranks highest and the rule can never say "all clear". Flags 5 of
#   8 healthy classes at threshold 0.50.
#
#   nb47, PERMUTATION-REFERENCED: fixes the scale problem, but still flags Web
#   (z = 7.5), Spoofing (6.5) and Recon (4.8) in the healthy environment. Those
#   are NOT spurious: CIC-IoT genuinely has drift, with S_cov,Web = 0.654. The
#   signal is correctly detecting movement. The problem is that MOVEMENT IS NOT
#   COVERAGE FAILURE.
#
# Step 2 already identified the operative quantity. Coverage is the target score
# distribution evaluated AT the source-calibrated threshold, and the fraction of
# movement realised at that threshold separates the two regimes cleanly: 0.99,
# 1.00 and 0.96 for classes that fail, 0.05 to 0.29 for classes that hold. A
# monitor should therefore measure threshold crossing, not movement.
#
# THE SIGNAL. For class c with source-calibrated quantile q_c, take the target
# flows PREDICTED c and compute the fraction whose score exceeds q_c. Under
# exchangeability that fraction is alpha by construction, so
#
#     signal_c = (exceedance rate among predicted-c) - alpha
#
# is zero when the quantile is respected and positive exactly when target mass has
# crossed above it. It is label-free (q_c comes from SOURCE labels, which we have;
# the target side needs none), absolute rather than relative, and on the same scale
# as the coverage deficit it is trying to anticipate.
#
# THE KNOWN BLIND SPOT PERSISTS BY CONSTRUCTION. A flow the shifted model
# confidently misroutes out of class c never enters the predicted-c set, so it
# cannot be measured. We expect the blind spot to remain and report it.
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
from conformal import conformal_q
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
MIN_SUPPORT=20; FAIL_THRESHOLD=0.05
print('ready | alpha', ALPHA)


Mounted at /content/drive
ready | alpha 0.05


In [2]:
# =============================================================================
# Cell 2 - the signal, plus the oracle it approximates, so the cost of the
# label-free substitution is measurable rather than assumed.
# =============================================================================
def aps_all(P):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

def rows_for(dataset, arch, classes, P_src, y_src, P_tgt, y_tgt, cov_lookup):
    Ss, St = aps_all(P_src), aps_all(P_tgt)
    yh_t = P_tgt.argmax(1)
    out=[]
    for ci,cn in enumerate(classes):
        # source-calibrated quantile from SOURCE labels (available by assumption)
        src_scores = Ss[y_src==ci, ci]
        if len(src_scores) < 5: continue
        q,_ = conformal_q(src_scores, ALPHA)
        if not np.isfinite(q): continue

        mp = yh_t==ci                      # target flows PREDICTED c  -> label-free
        mt = y_tgt==ci                     # target flows TRULY c      -> oracle only
        n_pred=int(mp.sum()); low = n_pred < MIN_SUPPORT

        # label-free: exceedance above the calibrated threshold among predicted-c
        exc_pred = float((St[mp,ci] > q).mean()) if not low else np.nan
        # oracle: the same quantity on the true class, which IS the coverage deficit
        exc_true = float((St[mt,ci] > q).mean()) if mt.sum()>=5 else np.nan

        out.append({'dataset':dataset,'arch':arch,'class':cn,
            'n_pred_target':n_pred,'n_true_target':int(mt.sum()),'low_support':bool(low),
            'q_source':float(q),
            'signal':exc_pred-ALPHA if exc_pred==exc_pred else np.nan,   # <-- the monitor
            'oracle':exc_true-ALPHA if exc_true==exc_true else np.nan,
            'exceedance_pred':exc_pred,'exceedance_true':exc_true,
            'coverage':cov_lookup.get(cn,np.nan)})
        out[-1]['undercoverage']=(1-ALPHA)-out[-1]['coverage'] if out[-1]['coverage']==out[-1]['coverage'] else np.nan
    return out

print('signal = (exceedance above q_c among target flows PREDICTED c) - alpha')
print('  zero when the calibrated quantile is respected; positive when mass has crossed it')
print('  oracle = the same on TRUE class, which equals the coverage deficit exactly')


signal = (exceedance above q_c among target flows PREDICTED c) - alpha
  zero when the calibrated quantile is respected; positive when mass has crossed it
  oracle = the same on TRUE class, which equals the coverage deficit exactly


In [3]:
# =============================================================================
# Cell 3 - compute across all four environments.
# =============================================================================
def check_classes(npz, assumed, tag):
    if 'classes' in npz.files:
        saved=[str(x) for x in npz['classes']]
        assert saved==list(assumed), f'{tag}: saved order {saved} != assumed'
        return saved
    return list(assumed)
def cov_map(f, rung=None):
    d=pd.read_csv(RD/f); d=d[(np.isclose(d.alpha,ALPHA))&(d.protocol=='SHC')]
    if 'feasible' in d.columns: d=d[d['feasible']]
    if rung is not None and 'rung' in d.columns: d=d[np.isclose(d['rung'],rung)]
    return d.groupby('class')['coverage'].mean().to_dict()

rows=[]; t0=time.time()
# ---- NSL-KDD ----
CL=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CL)}
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
y_sp=tr[tr.partition=='source_cal_pool']['label'].map(c2i).to_numpy()
y_te=te['label'].map(c2i).to_numpy()
assign=pd.read_parquet(config.PROC_DIR/'nslkdd_ladder_assignments.parquet')
IDX={(r,j,role):g['test_idx'].to_numpy() for (r,j,role),g in assign.groupby(['rung','realization','role'])}
REALS=sorted(assign[np.isclose(assign.rung,0.80)]['realization'].unique())[:5]
cm=cov_map('coverage_primary_nslkdd.csv',0.80)
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    d=np.load(f); P_sp=d['S_pool'].astype(np.float64); P_te=d['target'].astype(np.float64)
    for j in REALS:
        ev=IDX.get((0.80,j,'eval'))
        if ev is None or len(ev)==0: continue
        rows += rows_for('nslkdd',arch,CL,P_sp,y_sp,P_te[ev],y_te[ev],cm)
print(f'  nslkdd {sum(1 for r in rows if r["dataset"]=="nslkdd")} | {time.time()-t0:.0f}s')

# ---- UGR'16 ----
UGR=config.DATASETS_DIR/'ugr16'
us=pd.read_parquet(UGR/'july_week5.parquet'); ut=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (us,ut): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
us=us[us.label.isin(UK)].reset_index(drop=True); ut=ut[ut.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
def strat(df,fr,seed,col='label'):
    rg=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rg.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
us=us.assign(partition=strat(us,config.SPLIT_FRACTIONS,20260725).values)
y_usp=us[us.partition=='source_cal_pool']['label'].map(U2I).to_numpy()
y_utg=ut['label'].map(U2I).to_numpy()
cm=cov_map('coverage_primary_ugr16.csv')
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); d=np.load(f); cls=check_classes(d,UCL,'ugr16')
    rows += rows_for('ugr16',arch,cls,d['srcpool'].astype(np.float64),y_usp,
                     d['target'].astype(np.float64),y_utg,cm)
print(f'  ugr16 {sum(1 for r in rows if r["dataset"]=="ugr16")} | {time.time()-t0:.0f}s')

# ---- CIC-IoT-2023 ----
iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp=pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
ICL=json.loads((RD/'ciciot2023_model_record.json').read_text())['classes_canonical_order']
ic2i={c:i for i,c in enumerate(ICL)}; iot['y']=iot['family'].map(ic2i).astype(np.int64)
y_isp=iot.loc[iot.partition=='source_cal_pool','y'].to_numpy()
y_itg=iot.loc[iot.partition=='target_pool','y'].to_numpy()
cm=cov_map('coverage_primary_ciciot2023.csv',0.80)
for f in sorted((config.DATA_DIR/'ciciot_probs').glob('ciciot2023__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); d=np.load(f); cls=check_classes(d,ICL,'ciciot')
    rows += rows_for('ciciot2023',arch,cls,d['srcpool'].astype(np.float64),y_isp,
                     d['target'].astype(np.float64),y_itg,cm)
print(f'  ciciot2023 {sum(1 for r in rows if r["dataset"]=="ciciot2023")} | {time.time()-t0:.0f}s')

S=pd.DataFrame(rows)
dropped=S[S.coverage.isna()].groupby(['dataset','class']).size()
if len(dropped): print('\ndropped (no coverage entry):'); print(dropped.to_string())
S=S[S.coverage.notna()].reset_index(drop=True)
S['failing']=(S.undercoverage>FAIL_THRESHOLD).astype(int)
print(f'\n{len(S)} rows')


  nslkdd 600 | 12s
  ugr16 150 | 31s
  ciciot2023 240 | 71s

990 rows


In [4]:
# =============================================================================
# Cell 4 - does the threshold-referenced signal separate failing from healthy?
# =============================================================================
agg=S.groupby(['dataset','class'],as_index=False).agg(
    signal=('signal','mean'), oracle=('oracle','mean'),
    undercoverage=('undercoverage','mean'), failing=('failing','max'),
    n_pred=('n_pred_target','mean'), low_support=('low_support','max'))
ok=agg.signal.notna()
A=agg[ok].copy()
print(f"cells: {len(A)} | failing {int(A.failing.sum())} | healthy {int((1-A.failing).sum())}")
print("\nSIGNAL BY CLASS, sorted")
print(A.sort_values('signal',ascending=False)[
    ['dataset','class','signal','oracle','undercoverage','failing']].round(4).to_string(index=False))

r,_=stats.spearmanr(A.signal, A.undercoverage)
auc=roc_auc_score(A.failing, A.signal) if A.failing.nunique()>1 else np.nan
ro,_=stats.spearmanr(A.oracle.fillna(0), A.undercoverage)
print(f"\nlabel-free signal : rho {r:+.3f} | AUROC {auc:.3f}")
print(f"oracle (true class): rho {ro:+.3f}   <- the ceiling the label-free version approximates")

print("\nCOMPARISON WITH THE EARLIER DESIGNS (same measure, class level)")
prev=pd.read_csv(RD/'monitor_variant_comparison.csv')
print(prev.round(3).to_string(index=False))
print(f"  threshold-referenced (this notebook)   {r:+.3f}   {auc:.3f}")

print("\nFALSE ALARMS ON THE HEALTHY ENVIRONMENT")
FAIL_ENVS=[d for d,g in A.groupby('dataset') if g.failing.sum()>0]
neg=A[~A.dataset.isin(FAIL_ENVS)]; pos=A[A.dataset.isin(FAIL_ENVS)]
print(f"  healthy: {[d for d in A.dataset.unique() if d not in FAIL_ENVS]} ({len(neg)} classes)")
print(f"{'threshold':>10s} {'flagged/healthy':>16s} {'recall on failing':>18s}")
fa_rows=[]
for thr in [0.01,0.02,0.05,0.10,0.20]:
    f_=float((neg.signal>=thr).mean()) if len(neg) else np.nan
    rec=float((pos[pos.failing==1].signal>=thr).mean()) if (pos.failing==1).any() else np.nan
    fa_rows.append({'threshold':thr,'false_alarm':f_,'recall':rec})
    print(f"{thr:10.2f} {f_:16.3f} {rec:18.3f}")
F=pd.DataFrame(fa_rows)
clean=F[(F.false_alarm==0)&(F.recall>0)]
if len(clean):
    b=clean.iloc[0]
    print(f"\n  At signal >= {b.threshold:.2f} the rule raises ZERO false alarms on the healthy")
    print(f"  environment while retaining recall {b.recall:.3f} on the failing ones.")
    print("  Neither earlier design achieved this: the rank rule flags a fixed share of any")
    print("  environment, and the permutation-referenced rule flags real-but-harmless drift.")
else:
    print("\n  No threshold achieves zero false alarms with non-zero recall; report honestly.")


cells: 16 | failing 4 | healthy 12

SIGNAL BY CLASS, sorted
   dataset       class  signal  oracle  undercoverage  failing
ciciot2023         DoS -0.0469 -0.0408         0.0001        0
     ugr16  background -0.0490 -0.0428         0.0009        0
    nslkdd         DoS -0.0491  0.6224         0.5512        1
ciciot2023        DDoS -0.0497 -0.0465        -0.0000        0
    nslkdd      Normal -0.0498 -0.0215         0.0232        0
ciciot2023       Recon -0.0500 -0.0014         0.0003        0
ciciot2023      Benign -0.0500  0.0007        -0.0006        0
ciciot2023  BruteForce -0.0500 -0.0067        -0.0058        0
ciciot2023         Web -0.0500 -0.0015        -0.0019        0
ciciot2023    Spoofing -0.0500 -0.0026        -0.0028        0
ciciot2023       Mirai -0.0500 -0.0483        -0.0004        0
    nslkdd       Probe -0.0500  0.4372         0.3222        1
     ugr16         dos -0.0500 -0.0500        -0.0003        0
     ugr16 nerisbotnet -0.0500 -0.0290         0.0027     

In [ ]:
# =============================================================================
# Cell 5 - the blind spot, which this design does not fix and cannot.
# =============================================================================
print("THE BLIND SPOT PERSISTS BY CONSTRUCTION")
print("A flow the shifted model confidently misroutes out of class c never enters the")
print("predicted-c set, so the signal cannot see it. The gap between the label-free")
print("signal and the oracle measures exactly that loss.\n")
A['gap']=A.oracle-A.signal
print(A.sort_values('gap',ascending=False)[
    ['dataset','class','signal','oracle','gap','undercoverage']].round(4).head(10).to_string(index=False))
worst=A.loc[A.gap.idxmax()]
print(f"\n  largest gap: {worst['dataset']} {worst['class']}, signal {worst.signal:.3f} vs "
      f"oracle {worst.oracle:.3f}")
print("  The oracle sees the failure; the label-free proxy does not, because the failing")
print("  flows were assigned elsewhere. This is the identifiability limit of Section 5.13,")
print("  unchanged by the redesign and not claimed to be fixed by it.")
rg,_=stats.spearmanr(A.gap.fillna(0), A.undercoverage)
print(f"\n  gap vs undercoverage: rho {rg:+.3f} (the blind spot is worst where failure is worst)")

# save
S.to_csv(RD/'monitor_threshold_signals.csv', index=False)
A.to_csv(RD/'monitor_threshold_class_level.csv', index=False)
F.to_csv(RD/'monitor_threshold_falsealarm.csv', index=False)
(RD/'monitor_threshold_verdict.json').write_text(json.dumps({
 'design':'signal_c = (exceedance above the source-calibrated quantile among target flows '
          'PREDICTED c) - alpha; label-free, absolute, on the coverage scale',
 'motivation':'nb46 rank-based cannot say "all clear"; nb47 permutation-referenced flags '
              'real-but-harmless drift. Step 2 showed the operative quantity is movement '
              'AT the threshold, which this measures directly.',
 'n_cells':int(len(A)),'n_failing':int(A.failing.sum()),
 'rho':float(r),'auroc':float(auc),'oracle_rho':float(ro),
 'false_alarm_curve':F.round(4).to_dict('records'),
 'blind_spot':'unchanged: confidently misrouted flows never enter the predicted-c set, so '
              'the label-free signal cannot see them; the signal-to-oracle gap quantifies it',
 'power_caveat':f'{int(A.failing.sum())} failing classes; differences below roughly 0.15 '
                'AUROC are not resolvable'}, indent=2, default=str))
print('\nsaved threshold-monitor artefacts')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','step 7c: threshold-referenced monitor; measures crossing of the calibrated quantile rather than movement anywhere')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


THE BLIND SPOT PERSISTS BY CONSTRUCTION
A flow the shifted model confidently misroutes out of class c never enters the
predicted-c set, so the signal cannot see it. The gap between the label-free
signal and the oracle measures exactly that loss.

   dataset      class  signal  oracle    gap  undercoverage
    nslkdd        DoS -0.0491  0.6224 0.6715         0.5512
     ugr16     scan11 -0.0500  0.4552 0.5052         0.4150
    nslkdd      Probe -0.0500  0.4372 0.4872         0.3222
     ugr16     scan44 -0.0500  0.1465 0.1965         0.1510
ciciot2023     Benign -0.0500  0.0007 0.0507        -0.0006
ciciot2023      Recon -0.0500 -0.0014 0.0486         0.0003
ciciot2023        Web -0.0500 -0.0015 0.0485        -0.0019
ciciot2023   Spoofing -0.0500 -0.0026 0.0474        -0.0028
ciciot2023 BruteForce -0.0500 -0.0067 0.0433        -0.0058
    nslkdd     Normal -0.0498 -0.0215 0.0282         0.0232

  largest gap: nslkdd DoS, signal -0.049 vs oracle 0.622
  The oracle sees the failure; the 